In [1]:
from kadi_apy import KadiManager

In [6]:
manager = KadiManager(host="https://kadi4mat.iam-cms.kit.edu/", token="pat_97d6e147e888bfcc3b06eda10a1f4c1214616d9efb278b08")


In [7]:
import pandas as pd
import re
import os
import time

In [9]:
identifier = "micromagnetic-am-simulation"
collection= manager.collection(identifier = identifier , create = True)
collection.add_user(320, 'Editor')
print (collection)
collection.add_group_role(73, 'Editor')

collection 'micromagnetic-am-simulation' (id: 1428, identifier: 'micromagnetic-am-simulation')


<Response [409]>

- Create a collection to put all of our records in a single Collection.
- print the collection to get the id; we need the id while we are planning to add the records in collection we need the id number

- add the user and their role

In [28]:
csv_path = 'C:/Users/nabod/average_values.csv'

# Try different encodings or skip problematic rows if needed
try:
    df = pd.read_csv(csv_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, encoding='latin-1')

# Display the DataFrame
print(df)


    v [mm s-1]  P [W]  b [¦Ìm]  d [¦Ìm]  avg_pe (domain)  \
0          100   30.0   92.269   51.779            0.030   
1          150   30.0   23.077    8.338            0.022   
2           50   30.0  190.726  101.453            0.039   
3           75   30.0  138.123   80.798            0.036   
4          100   35.0  147.981   87.714            0.036   
5          100   32.5  124.628   74.797            0.033   
6          150   35.0   91.833   51.781            0.027   
7           50   25.0   65.910   29.846            0.029   
8           75   25.0   26.444    8.410            0.028   
9          100   25.0   12.355    6.634            0.025   
10         125   25.0    0.000    0.000            0.016   
11         150   25.0    0.000    0.000            0.016   
12         100   27.5   67.643   30.950            0.028   
13         125   30.0   67.694   31.699            0.026   
14          50   27.5  165.682   90.840            0.039   
15          50   32.5  222.926  114.669 

- create a data frame from a csv file

In [29]:
# Select the desired columns
new_df = df.loc[:, ['v [mm s-1]', 'P [W]']]

# Display the new DataFrame
print(new_df)

    v [mm s-1]  P [W]
0          100   30.0
1          150   30.0
2           50   30.0
3           75   30.0
4          100   35.0
5          100   32.5
6          150   35.0
7           50   25.0
8           75   25.0
9          100   25.0
10         125   25.0
11         150   25.0
12         100   27.5
13         125   30.0
14          50   27.5
15          50   32.5
16         125   35.0
17         125   27.5
18          75   32.5
19          75   35.0


- sort the data frame as we want.we want to create the record by using Power and Scan Speed

In [30]:
#Rename the columns
new_df = new_df.rename(columns={'v [mm s-1]': 'Scan Speed', 'P [W]': 'Power'})

# Add additional columns
new_df['Power Unit'] = 'Watt'
new_df['Scan Speed Unit'] = 'mm s-1'

# Reorder the columns
new_df = new_df[['Power', 'Power Unit', 'Scan Speed', 'Scan Speed Unit']]

# Display the updated DataFrame
print(new_df)


    Power Power Unit  Scan Speed Scan Speed Unit
0    30.0       Watt         100          mm s-1
1    30.0       Watt         150          mm s-1
2    30.0       Watt          50          mm s-1
3    30.0       Watt          75          mm s-1
4    35.0       Watt         100          mm s-1
5    32.5       Watt         100          mm s-1
6    35.0       Watt         150          mm s-1
7    25.0       Watt          50          mm s-1
8    25.0       Watt          75          mm s-1
9    25.0       Watt         100          mm s-1
10   25.0       Watt         125          mm s-1
11   25.0       Watt         150          mm s-1
12   27.5       Watt         100          mm s-1
13   30.0       Watt         125          mm s-1
14   27.5       Watt          50          mm s-1
15   32.5       Watt          50          mm s-1
16   35.0       Watt         125          mm s-1
17   27.5       Watt         125          mm s-1
18   32.5       Watt          75          mm s-1
19   35.0       Watt

- renamed the column and added the full form of the unit (optional).did this for consistency

In [ ]:

start_time = time.time()
#files_directory = r"C:\Users\nabod\mesostructures"
files_directory = r"C:\Users\nabod\Downloads\sampled_point_data"
# Loop through the rows of the DataFrame and create the records with metadata
for index, row in new_df.iterrows():
    cycle_time = time.time()
    power = str(row['Power'])
    power_unit = row['Power Unit']
    scan_speed = str(row['Scan Speed'])
    scan_speed_unit = row['Scan Speed Unit']
    
    # Split the power value before and after the decimal point
    power_split = power.split('.')
    power_before_decimal = power_split[0]
    power_after_decimal = power_split[1] if len(power_split) > 1 else '0'
    
    # Create the identifier based on the power and scan speed values
    identifier = f"fe_ni_am_{power_before_decimal}-{power_after_decimal}_{scan_speed}"
    
    # Create the record
    record = manager.record(identifier=identifier, create=True)
    data = {
        'key': 'Process Parameters',
        'type': 'dict',
        'value': [
            {'key': 'Power', 'type': 'float', 'unit': power_unit, 'value': float(power)},
            {'key': 'Scan Speed', 'type': 'float', 'unit': scan_speed_unit, 'value': float(scan_speed)}
        ]
    }
    
    # Add metadata to the record
    record.add_metadata(data)
    # add user 
    record.add_user(user_id=320, role_name='Editor')

    for file_name in os.listdir(files_directory):
        # Extract power and scan speed from the file name using regular expressions
        matches = re.findall(r"(\d+\.*\d+)", file_name)
        #print(matches)
        file_power = float(matches[0])
        #print(file_power)
        file_scan_speed = float(matches[1])
        #print(file_scan_speed)
    
        # Compare file power and scan speed with the values from the DataFrame
        if file_power == float(power) and file_scan_speed == float(scan_speed):
            file_path = os.path.join(files_directory, file_name)
            print(f"Starting Upload {file_name}")          
            # Upload the file to the record
            record.upload_file(file_path)
            print(f"Uploaded {file_name}")
            cycle_end_time = time.time()
            print(f"Time Elapsed: {cycle_end_time - cycle_time}")
            
     # Add the record to the collection
    collection_id = 1428  # Replace with your collection ID
    record.add_collection_link(collection_id)
    

            

end_time = time.time()
    
print(f"Total Elapsed Time: {end_time - start_time}")

- **re** is for regular expression
- start and end time for gettiing the time for creating the records
- Loop through the rows of the DataFrame and create the records with metadata
        - as the identifiers are not getting the **.** ,then we have to split the values of power..for the scan speed we have no decimal point, but for Power it has some cases 30.5, 75.5. We can use any other identifiers also, but our target is to from the identifier we can get a idea which power and scan speed it is.
- so we made all of those as String and then split it before and after decimal point.
-  then before decimal it listed as 0 and after decimal 1 and in between a -
- now use the identifier for all records and 20 records are created
- then defined the data and added as metadata
- now our target is to upload a lot of files naming according to their Power and Scan Speed values. But the problem is that the files are not naming as same as the data frame, the names of the files are different but the number are same but numbers are written like 30-70,25.5-100 like this not 30.0-70.0, 25.5-100.0.. so for finding the names and matching with the same identifier is a challenging job.
- here we used the Regular expression which will find the repeated character and also the decimal(be sure for the logic of the re.findall (r"(......)" this line.
- it will start to scan for all of these files according the logic;we defined our files_directory; starts to scan all of the files whose name are matches with these two digits and then match and upload and it again start to scan..if nothing match then go to the next one.
- then make the file power and file scan as float then match and start to upload
- at the end print the time
- add the records to the collection by using collection id
